In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from pathlib import Path
from typing import Callable

from src import SimulationConfig, EigenvalueSimulator

In [31]:
# Working functions
def linear_scaling(a: float, b: float) -> Callable[[int], float]:
    def p_func(n: int) -> float:
            return a * n - b

    p_func.__name__ = f"linear_{a}n_-_{b}"

def id(n: int) -> float:
    return n

def power_scaling(beta: float) -> Callable[[int], float]:
    def p_func(n: int) -> float:
        return n ** beta

    p_func.__name__ = f"power_{beta}"
    return p_func


def log(n: int) -> float:
    return np.log(n)


def loglog(n: int) -> float:
    return np.log1p(n)

def constant(p: float) -> Callable[[int], float]:
    def p_func(n: int) -> float:
        return p

    p_func.__name__ = f"constant_{p}"

In [53]:
CSV_PATH  = "../data/simulation_results.csv"

# Help function for visualization
def simulate(p_func: Callable[[int], float]) -> None:
    for n in range(5, 300, 5):
        config = SimulationConfig(
            n=n,
            p_func=p_func,
            epsilon=0.05,
            min_samples=0,
            batch_size=5000,
            sweep_tag=f"{p_func.__name__} for [5, 300]"
        )

        print(f"Launching simulation  sweep={config.sweep_tag}  n={config.n}")

        EigenvalueSimulator(config).run(
            log_to=Path("../data/simulation_results.csv"),
        )

def load_sweep(csv_path: str, sweep_tag: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df[df["sweep_tag"] == sweep_tag].sort_values("n")

def power_law(n, C, a):
    return C * n ** a

def fit_power_law(n, y, yerr):
    popt, pcov = curve_fit(power_law, n, y, sigma=yerr, absolute_sigma=True)
    perr = np.sqrt(np.diag(pcov))
    return popt, perr

def plot_fit(n, y, n_fit, y_fit, C, a, perr, sweep_tag: str):
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.scatter(n, y, label="simulation")
    ax.plot(n_fit, y_fit, label=f"fit: {C:.3f} · n^{a:.3f}  (±{perr[1]:.3f})")

    ax.set_xlabel("n")
    ax.set_ylabel("mean largest eigenvalue")
    ax.set_title(f"Sweep: {sweep_tag}")
    ax.legend()

def visualize(sweep_tag: str) -> None:
    df   = load_sweep(CSV_PATH, sweep_tag)

    n    = df["n"].to_numpy()
    y    = df["mean_eigenvalue"].to_numpy()
    yerr = df["_std_error"].to_numpy()

    (C, a), perr = fit_power_law(n, y, yerr)

    n_fit = np.linspace(n.min(), n.max())
    y_fit = power_law(n_fit, C, a)

    plot_fit(n, y, n_fit, y_fit, C, a, perr, sweep_tag)

    print(f"C = {C:.4f} ± {perr[0]:.4f}")
    print(f"a = {a:.4f} ± {perr[1]:.4f}")


In [49]:
# Run simulations
simulate(id)
simulate(log)
simulate(loglog)

for b in range(1, 10):
    simulate(power_scaling(b/10))

# Assuming that n are big, b does not make impact
for a in range(1, 10):
    simulate(linear_scaling(a/10, 0))

for p in range(1, 10):
    simulate(constant(p/10))


Launching simulation  sweep=id for [5, 300]  n=5
Launching simulation  sweep=id for [5, 300]  n=10
Launching simulation  sweep=id for [5, 300]  n=15
Launching simulation  sweep=id for [5, 300]  n=20
Launching simulation  sweep=id for [5, 300]  n=25
Launching simulation  sweep=id for [5, 300]  n=30
Launching simulation  sweep=id for [5, 300]  n=35
Launching simulation  sweep=id for [5, 300]  n=40
Launching simulation  sweep=id for [5, 300]  n=45
Launching simulation  sweep=id for [5, 300]  n=50
Launching simulation  sweep=id for [5, 300]  n=55
Launching simulation  sweep=id for [5, 300]  n=60
Launching simulation  sweep=id for [5, 300]  n=65
Launching simulation  sweep=id for [5, 300]  n=70
Launching simulation  sweep=id for [5, 300]  n=75
Launching simulation  sweep=id for [5, 300]  n=80
Launching simulation  sweep=id for [5, 300]  n=85
Launching simulation  sweep=id for [5, 300]  n=90
Launching simulation  sweep=id for [5, 300]  n=95
Launching simulation  sweep=id for [5, 300]  n=100


In [ ]:
# Do visualizations
SWEEP_TAG = f"{id.__name__} for [5, 300]"
visualize(SWEEP_TAG)

SWEEP_TAG = f"{log.__name__} for [5, 300]"
visualize(SWEEP_TAG)

SWEEP_TAG = f"{loglog.__name__} for [5, 300]"
visualize(SWEEP_TAG)

for b in range(1, 10):
    SWEEP_TAG = f"{power_scaling(b/10).__name__} for [5, 300]"
    simulate(SWEEP_TAG)

# Assuming that n are big, b does not make impact
for a in range(1, 10):
    SWEEP_TAG = f"{linear_scaling(a/10, 0).__name__} for [5, 300]"
    simulate(SWEEP_TAG)

for p in range(1, 10):
    SWEEP_TAG = f"{constant(p/10).__name__} for [5, 300]"
    visualize(SWEEP_TAG)

plt.show()